In [ ]:
from __future__ import (absolute_import, division,
                        print_function, unicode_literals)

import warnings
warnings.simplefilter('ignore')

# general purpose packages
import pandas as pd
import numpy as np
import os
import json
import time
import re
import csv
import subprocess
import sys

import scipy.stats as stats
import statsmodels.stats as smstats
from statsmodels.stats.multitest import multipletests

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from dotenv import load_dotenv
from pathlib import Path

from multiprocessing import Process, Manager, Pool
import multiprocessing
from functools import partial

from collections import Counter

import seaborn as sns; sns.set()

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
matplotlib.rcParams['backend'] = "Qt5Agg"
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter

from IPython.display import display, Image

from adjustText import adjust_text
import builtins
%matplotlib inline

# for normalization
from sklearn.linear_model import QuantileRegressor

# for survival analysis
import sklearn
from sklearn import set_config

from statsmodels.regression.quantile_regression import QuantReg

# for working with yaml files
import ruamel.yaml

import itertools

In [ ]:
def get_pvalue_star(pval, thr=0.05):
    if thr == 0.05:
        if pval < 0.001:
            return "***"
        elif pval < 0.01:
            return "**"
        elif pval < 0.05:
            return "*"
        else:
            return ""
    elif thr == 0.1:
        if pval < 0.001:
            return "***"
        elif pval < 0.01:
            return "**"
        elif pval < 0.1:
            return "*"
        else:
            return ""

In [ ]:
# 1. Load the environment variables
load_dotenv("TemperatureDependentCelegansAging_Spang.scicore.env",override=True)

# 2. Reconstruct the subdirs dictionary
subdirs = {
    "lab_group_dir": os.getenv("LAB_GROUP_DIR"),
    "raw_sequencing_data_dir": os.getenv("RAW_SEQUENCING_DATA_DIR"),
    "main_project_dir": os.getenv("MAIN_PROJECT_DIR"),
    "wf_dir": os.getenv("WF_DIR"),
    "UCSCtracks_dir": os.getenv("UCSC_TRACKS_DIR"),
    "UCSCtracks_trackfiles_dir": os.getenv("UCSC_TRACKFILES_DIR"),
    "UCSCtracks_trackhubs_dir": os.getenv("UCSC_TRACKHUBS_DIR"),
    "celegans_annotation_dir": os.getenv("CELEGANS_ANNOTATION_DIR"),
    "shared_project_dir": os.getenv("SHARED_PROJECT_DIR"),
    "temp_dir": os.getenv("TEMP_DIR"),
    "slurm_dir": os.getenv("SLURM_DIR"),
    "slurm_scripts_dir": os.getenv("SLURM_SCRIPTS_DIR"),
    "figures_dir": os.getenv("FIGURES_DIR"),
    "tables_dir": os.getenv("TABLES_DIR"),
    "fastq_dir": os.getenv("FASTQ_DIR"),
    "metadata_dir": os.getenv("METADATA_DIR"),
    "wf_runs_dir": os.getenv("WF_RUNS_DIR"),
    "configs_dir": os.getenv("CONFIGS_DIR"),
}

# 3. Reconstruct the file_paths dictionary
file_paths = {
    "celegans_genome_file": os.getenv("CELEGANS_GENOME_FILE"),
    "celegans_chrom_sizes_file": os.getenv("CELEGANS_CHROM_SIZES_FILE"),
    "celegans_annotation_file": os.getenv("CELEGANS_ANNOTATION_FILE"),
    "celegans_polyAsite_atlas": os.getenv("CELEGANS_POLYASITE_ATLAS"),
}

# 4. Safely create all subdirectories
# Using os.makedirs is highly preferred over os.system('mkdir -p')
# because it avoids opening a subshell and handles permissions gracefully in pure Python.
for path in subdirs.values():
    if path:  # Safety check to ensure the variable was actually found in the .env
        os.makedirs(path, exist_ok=True)

print("Environment loaded and directories verified.")

# Prepare start samples

In [ ]:
paths_to_look = [
subdirs['raw_sequencing_data_dir']+'20260228*', # these are the newest long RNA data, based on biotinylation, made with circularization protocol, they are single-end
]

fastq_file_paths = []
for path_ in paths_to_look:
    command = """find """+path_+""" -name '*.fastq.gz' > """+subdirs['temp_dir']+"""fastq_file_paths.tsv"""
    out = subprocess.check_output(command, shell=True)
    tmp = pd.read_csv(subdirs['temp_dir']+'fastq_file_paths.tsv',delimiter="\t",
                                   index_col=None,header=None)
    fastq_file_paths.append(tmp)
fastq_file_paths = pd.concat(fastq_file_paths).reset_index(drop=True)
fastq_file_paths.columns = ['path']

fastq_file_paths['sample'] = fastq_file_paths['path'].str.split('/',expand=True).iloc[:,-1].str.split('.',expand=True)[0]

In [ ]:
# Create symbolink link copies for all experimental fastq files

fastq_file_paths['start_file_path'] = subdirs['fastq_dir']+fastq_file_paths['sample']+'.fastq.gz'

# uncomment if need to re-generate, otherwise re-creation may invoke unnecessary Snakemake re-execution
for index, row in fastq_file_paths.iterrows():
    command = 'rm -f '+row['start_file_path']+' && ln -f -s '+row['path']+' '+row['start_file_path']
    out = subprocess.check_output(command, shell=True)

fastq_file_paths = fastq_file_paths.drop(['path'],axis=1).rename(columns={'start_file_path':'path'})

In [ ]:
# lanes should be aligned separately, but then merged together
fastq_file_paths['lane'] = fastq_file_paths.apply(lambda x:int(x['sample'].split('_L00')[1].split('_')[0]),1)
fastq_file_paths['merged_sample'] = fastq_file_paths.apply(lambda x:x['sample'].split('_')[0],1) # GFB... is in general consistent naming

start_samples = fastq_file_paths.copy()

In [ ]:
start_samples['fq1'] = start_samples['path']
start_samples['fq1_3p'] = 'AGATCGGAAGAG' # forcely put Illumina universal adapter

In [ ]:
WF_version = 'v1_0_0'

dir_path = subdirs['wf_runs_dir']+WF_version+'/'
command = 'mkdir -p '+dir_path
out = subprocess.check_output(command, shell=True)

start_samples.to_csv(dir_path+'start_samples.tsv', sep=str('\t'),header=True,index=None,quoting=csv.QUOTE_NONE)

In [ ]:
# all merged_samples should have TWO samples corresponding to two lanes:
start_samples['t']=1
start_samples.groupby(['merged_sample']).agg({'t':np.sum})

# Prepare .yaml config file and run WF

create conda environment with snakemake and install SLURM executor

run from the login node on HPC cluster:

```bash
conda create -c conda-forge -c bioconda -n snakemake snakemake
conda activate snakemake
pip install snakemake-executor-plugin-slurm
```

In [ ]:
organism = 'celegans'

gtf_chrs = pd.read_csv(file_paths[organism+'_annotation_file'],delimiter="\t",
                                   index_col=None,header=None,usecols = [0],skiprows=5)
chromosome_list = list(gtf_chrs[0].unique())

In [ ]:
# load default rule_config, modify it and save
WF_version = 'v1_0_0'
organism = 'celegans'

yaml = ruamel.yaml.YAML()
yaml.preserve_quotes = True
with open(subdirs['wf_dir']+'config.yaml') as f_read:
    data = yaml.load(f_read)

data['samples_file'] = subdirs['wf_runs_dir']+WF_version+'/start_samples.tsv'
data['output_dir'] = subdirs['wf_runs_dir']+WF_version+'/output/'
data['local_log'] = subdirs['wf_runs_dir']+WF_version+'/output/local_log/'
data['cluster_log'] = subdirs['wf_runs_dir']+WF_version+'/output/cluster_log/'

data['organism'] = organism
data['genome_file'] = file_paths[organism+'_genome_file']
data['gtf_file'] = file_paths[organism+'_annotation_file']

data['chromosomes'] = ' '.join(chromosome_list)

for dir_path in [data['output_dir'],data['local_log'],data['cluster_log']]:
    command = 'mkdir -p '+dir_path
    out = subprocess.check_output(command, shell=True)

with open(subdirs['wf_runs_dir']+WF_version+'/modified_config.yaml','w') as f_write:     
    yaml.dump(data, f_write)

WF_step = "prepare-faster-se"

command = """snakemake \
--snakefile """+subdirs['wf_dir']+"""Snakefile-"""+WF_step+""" \
--scheduler greedy \
--configfile """+subdirs['wf_runs_dir']+WF_version+'/modified_config.yaml'+""" \
--printshellcmds \
--software-deployment-method conda apptainer \
--conda-frontend mamba \
--apptainer-args "--bind """+subdirs['wf_dir']+','+subdirs['lab_group_dir']+','+subdirs['raw_sequencing_data_dir']+"""" \
--executor slurm \
--profile """+subdirs['wf_dir']+'profile'+""" \
--nolock \
-np"""

print(command)
# remember to run the command when "snakemake" conda env is activated!
# NOTE - there is an argument "-np" in the end - that results in a mock run, creating a DAG of jobs, to check the correctness of the planned run.
# To actually run, just delete the "-np" from the end of the command.

execute the command above from login node

snakemake automatically submits and monitors computational jobs to SLURM queue manager

The results of all workflow steps will be necessary to conduct further data analysis below, e.g. analyzing gene expression and alternative polyadenylation (APA)

# Analysis of read mapping stats - ongoing

In [ ]:
WF_version = "v1_0_0"
organism = "celegans"
dir_path = subdirs["wf_runs_dir"] + WF_version + "/"

os.system(
    """find """
    + dir_path
    + "output/mapping_stats/"
    + """ -name '*.mapping_stats.txt' > """
    + subdirs["temp_dir"]
    + """mapping_stats.files.txt"""
)
os.system(
    """find """
    + dir_path
    + "output/mapping_stats/"
    + """ -name '*.success' > """
    + subdirs["temp_dir"]
    + """mapping_stats.success.files.txt"""
)

mapping_stats_files = pd.read_csv(
    subdirs["temp_dir"] + "mapping_stats.files.txt",
    delimiter="\t",
    index_col=None,
    header=None,
)
mapping_stats_success_files = pd.read_csv(
    subdirs["temp_dir"] + "mapping_stats.success.files.txt",
    delimiter="\t",
    index_col=None,
    header=None,
)

mapping_stats_files["merged_sample"] = mapping_stats_files.apply(
    lambda x: x[0].split("/")[-1].replace(".mapping_stats.txt", ""), 1
)
mapping_stats_success_files["merged_sample"] = mapping_stats_success_files.apply(
    lambda x: x[0].split("/")[-1].replace(".success", ""), 1
)

mapping_stats_files = mapping_stats_files.loc[
    mapping_stats_files["merged_sample"].isin(
        list(mapping_stats_success_files["merged_sample"].unique())
    )
].reset_index(
    drop=True
)  # look only at files with success flag

start_samples = pd.read_csv(
    dir_path + "start_samples.tsv", delimiter="\t", index_col=None, header=0
)
metadata_df = get_metadata_with_labels_and_colors(start_samples)

sel_files = pd.merge(
    mapping_stats_files,
    metadata_df.drop_duplicates("merged_sample"),
    how="right",
    on=["merged_sample"],
)
samples_list = list(sel_files["merged_sample"])  # list of samples

In [ ]:
sub_sel_files = sel_files.copy()

i = 0
res = []
for index, row in sub_sel_files.iterrows():
    # parse custom format
    tmp = pd.read_csv(row[0], delimiter="\t", index_col=None, header=None)
    df_chunk_source, cur_col_name = [], ""
    res_df = None
    for elem in tmp[0].values:
        if elem.startswith(">"):
            # create a df from the previous chunk
            if len(df_chunk_source) > 0 and cur_col_name != "":
                df_chunk = pd.DataFrame(df_chunk_source, columns=[cur_col_name])
                df_chunk[cur_col_name] = df_chunk[cur_col_name].astype("int")
                df_chunk["t"] = 1
                if res_df is None:
                    res_df = df_chunk[[cur_col_name, "t"]]
                else:
                    res_df = pd.merge(
                        res_df, df_chunk[[cur_col_name, "t"]], how="inner", on="t"
                    )
            cur_col_name = elem[1:]
            df_chunk_source = []
        else:
            df_chunk_source.append(elem)
    if len(df_chunk_source) > 0 and cur_col_name != "":
        df_chunk = pd.DataFrame(df_chunk_source, columns=[cur_col_name])
        df_chunk[cur_col_name] = df_chunk[cur_col_name].astype("int")
        df_chunk["t"] = 1
        if res_df is None:
            res_df = df_chunk[[cur_col_name, "t"]]
        else:
            res_df = pd.merge(
                res_df, df_chunk[[cur_col_name, "t"]], how="inner", on="t"
            )
    res_df["sample"] = row["sample"]
    res.append(res_df.drop(["t"], axis=1))
    if i % 2 == 0 and i != 0:
        print(str(i) + " done")
    i = i + 1
mapping_stats_df = pd.concat(res).reset_index(drop=True)
mapping_stats_df = pd.merge(sel_files, mapping_stats_df, how="left", on="sample")
mapping_stats_df["number of MM reads; genomic mapping"] = (
    mapping_stats_df["total number of mapped reads; genomic mapping"]
    - mapping_stats_df["number of UM reads; genomic mapping"]
)
mapping_stats_df["number of MM reads; UMI dedup"] = (
    mapping_stats_df["total number of mapped reads; UMI dedup"]
    - mapping_stats_df["number of UM reads; UMI dedup"]
)

numb_cols = list(mapping_stats_df.columns)[-6:]
for elem in numb_cols:
    mapping_stats_df[elem + "; mln"] = np.round(mapping_stats_df[elem] / 10**6, 2)

mapping_stats_df["perc_of_UM; genomic mapping"] = (
    mapping_stats_df["number of UM reads; genomic mapping"]
    / mapping_stats_df["total number of mapped reads; genomic mapping"]
    * 100
)
mapping_stats_df["perc_of_UM; UMI dedup"] = (
    mapping_stats_df["number of UM reads; UMI dedup"]
    / mapping_stats_df["total number of mapped reads; UMI dedup"]
    * 100
)

In [ ]:
features = [
    "total number of mapped reads; genomic mapping; mln",
    "number of UM reads; genomic mapping; mln",
    "total number of mapped reads; UMI dedup; mln",
    "number of UM reads; UMI dedup; mln",
]
palette = ["black", "green", "grey", "royalblue"]

sns.set(font_scale=1)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=True, figsize=(4, 8))

for i, feature in enumerate(features):

    x_feature, y_feature = feature, "sample_label_within_experiment"

    ax = sns.pointplot(
        data=mapping_stats_df, x=x_feature, y=y_feature, color=palette[i], label=feature
    )
    ax.legend(bbox_to_anchor=(1.05, 1.0), loc=2, borderaxespad=0.0, title="", ncols=1)
    # ax.set_yticklabels(mapping_stats_df['sample_label_within_experiment'])
    ax.tick_params(left=True, bottom=True)
ax.set(ylabel="", xlabel="# reads, mln")
out = subprocess.check_output(
    "mkdir -p " + subdirs["figures_dir"] + "mapping_stats/", shell=True
)
fig.savefig(
    subdirs["figures_dir"] + "mapping_stats/read_mapping.read_numbers.png",
    bbox_inches="tight",
    dpi=300,
)
fig.savefig(
    subdirs["figures_dir"] + "mapping_stats/read_mapping.read_numbers.pdf",
    bbox_inches="tight",
    dpi=300,
)

In [ ]:
mapping_stats_df['number of UM reads; UMI dedup; mln'].quantile(0.5)

In [ ]:
mapping_stats_df[['sample','number of UM reads; UMI dedup; mln']].sort_values('number of UM reads; UMI dedup; mln')

In [ ]:
features = ["perc_of_UM; genomic mapping", "perc_of_UM; UMI dedup"]
palette = ["green", "royalblue"]

sns.set(font_scale=1)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=True, figsize=(4, 8))

for i, feature in enumerate(features):

    x_feature, y_feature = feature, "sample_label_within_experiment"

    ax = sns.pointplot(
        data=mapping_stats_df, x=x_feature, y=y_feature, color=palette[i], label=feature
    )
    ax.legend(bbox_to_anchor=(1.05, 1.0), loc=2, borderaxespad=0.0, title="", ncols=1)
    ax.tick_params(left=True, bottom=True)
ax.set(ylabel="", xlabel="% of mapped reads")
out = subprocess.check_output(
    "mkdir -p " + subdirs["figures_dir"] + "mapping_stats/", shell=True
)
fig.savefig(
    subdirs["figures_dir"] + "mapping_stats/read_mapping.perc_of_UM.png",
    bbox_inches="tight",
    dpi=300,
)
fig.savefig(
    subdirs["figures_dir"] + "mapping_stats/read_mapping.perc_of_UM.pdf",
    bbox_inches="tight",
    dpi=300,
)